# 02 - Flickr API scraping

## Aim
This notebook collects Flickr images related to brutalist architecture using the Flickr API.

## Outputs
- image metadata
- image URLs
- tags
- owner
- CSV for later processing

In [14]:
import requests
import pandas as pd
from tqdm import tqdm
import time

API_KEY = "62d32f31136292976d51b2d06d9372f2"

base_url = "https://www.flickr.com/services/rest/"

In [15]:
def search_photos(api_key, text, per_page=100, page=1):
    params = {
        "method": "flickr.photos.search",
        "api_key": api_key,
        "text": text,
        "per_page": per_page,
        "page": page,
        "format": "json",
        "nojsoncallback": 1,
        "content_type": 1,   # photos only
        "media": "photos",
        "sort": "relevance",
        "extras": "tags,url_o,url_l,url_c,url_z,owner_name,date_upload"
    }
    r = requests.get(base_url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [16]:
keyword = "brutalist architecture"
data = search_photos(API_KEY, keyword, per_page=5, page=1)
data.keys()

dict_keys(['photos', 'stat'])

In [17]:
data["photos"].keys()

dict_keys(['page', 'pages', 'perpage', 'total', 'photo'])

In [18]:
data["photos"]["photo"][:2]

[{'id': '43539066400',
  'owner': '135414206@N06',
  'secret': 'b757058e13',
  'server': '1922',
  'farm': 2,
  'title': 'Brutalist Architecture',
  'ispublic': 1,
  'isfriend': 0,
  'isfamily': 0,
  'dateupload': '1539665544',
  'ownername': 'Leipzig_trifft_Wien',
  'tags': 'brutalist architecture blackandwhite black white concrete building city urban style',
  'upgrade_sizes': ['h', 'k', 'o'],
  'url_l': 'https://live.staticflickr.com/1922/43539066400_b757058e13_b.jpg',
  'height_l': 682,
  'width_l': 1024,
  'url_c': 'https://live.staticflickr.com/1922/43539066400_b757058e13_c.jpg',
  'height_c': 533,
  'width_c': 800,
  'url_z': 'https://live.staticflickr.com/1922/43539066400_b757058e13_z.jpg',
  'height_z': 427,
  'width_z': 640},
 {'id': '47555029381',
  'owner': '34608298@N05',
  'secret': 'b49218bb8d',
  'server': '7867',
  'farm': 8,
  'title': 'Brutalist Architecture',
  'ispublic': 1,
  'isfriend': 0,
  'isfamily': 0,
  'dateupload': '1554633168',
  'ownername': 'Cat Girl 00

In [19]:
def extract_photo_records(photo_list, source_keyword):
    records = []

    for p in photo_list:
        image_url = (
            p.get("url_o") or
            p.get("url_l") or
            p.get("url_c") or
            p.get("url_z")
        )

        # 如果 API 没给 URL，就手动拼
        if not image_url:
            farm = p.get("farm")
            server = p.get("server")
            photo_id = p.get("id")
            secret = p.get("secret")

            if farm and server and photo_id and secret:
                image_url = f"https://farm{farm}.staticflickr.com/{server}/{photo_id}_{secret}.jpg"

        records.append({
            "photo_id": p.get("id"),
            "owner": p.get("owner"),
            "owner_name": p.get("ownername"),
            "title": p.get("title"),
            "tags": p.get("tags"),
            "date_upload": p.get("dateupload"),
            "image_url": image_url,
            "keyword": source_keyword,
            "source": "flickr"
        })

    return records

In [20]:
df_flickr["image_url"].head(10)

0    https://live.staticflickr.com/1922/43539066400...
1    https://live.staticflickr.com/7867/47555029381...
2    https://live.staticflickr.com/8251/29402571096...
3    https://live.staticflickr.com/4339/36546780916...
4    https://live.staticflickr.com/4419/36381698731...
5    https://live.staticflickr.com/7899/47555030591...
6    https://live.staticflickr.com/4066/4406268251_...
7    https://live.staticflickr.com/7865/47555028081...
8    https://live.staticflickr.com/7586/16593973533...
9    https://live.staticflickr.com/5323/16999172930...
Name: image_url, dtype: object

In [21]:
records = extract_photo_records(data["photos"]["photo"], keyword)
df_test = pd.DataFrame(records)
df_test.head()

,photo_id,owner,owner_name,title,tags,date_upload,image_url,keyword,source
0,43539066400,135414206@N06,Leipzig_trifft_Wien,Brutalist Architecture,brutalist architecture blackandwhite black whi...,1539665544,https://live.staticflickr.com/1922/43539066400...,brutalist architecture,flickr
1,47555029381,34608298@N05,Cat Girl 007,Brutalist Architecture,abstract architecture brutalism brutalist brut...,1554633168,https://live.staticflickr.com/7867/47555029381...,brutalist architecture,flickr
2,29402571096,21224104@N02,Suzanne Neubauer,Brutalist architecture,monochromatic blackandwhite brutalist architec...,1472949205,https://live.staticflickr.com/8251/29402571096...,brutalist architecture,flickr
3,36381698731,73331227@N05,michael.veltman,Brutalist,bruatlist architecture chicago illinois,1502539836,https://live.staticflickr.com/4419/36381698731...,brutalist architecture,flickr
4,47555030591,34608298@N05,Cat Girl 007,Brutalist Architecture,abstract architecture brutalism brutalist brut...,1554633168,https://live.staticflickr.com/7899/47555030591...,brutalist architecture,flickr


In [22]:
keyword = "brutalist architecture"

all_records = []

for page in range(1, 4):  # 先抓前3页测试
    print(f"Fetching page {page}")
    data = search_photos(API_KEY, keyword, per_page=100, page=page)
    photo_list = data["photos"]["photo"]
    records = extract_photo_records(photo_list, keyword)
    all_records.extend(records)
    time.sleep(1)

df_flickr = pd.DataFrame(all_records)
df_flickr.head()

Fetching page 1
Fetching page 2
Fetching page 3


,photo_id,owner,owner_name,title,tags,date_upload,image_url,keyword,source
0,43539066400,135414206@N06,Leipzig_trifft_Wien,Brutalist Architecture,brutalist architecture blackandwhite black whi...,1539665544,https://live.staticflickr.com/1922/43539066400...,brutalist architecture,flickr
1,47555029381,34608298@N05,Cat Girl 007,Brutalist Architecture,abstract architecture brutalism brutalist brut...,1554633168,https://live.staticflickr.com/7867/47555029381...,brutalist architecture,flickr
2,29402571096,21224104@N02,Suzanne Neubauer,Brutalist architecture,monochromatic blackandwhite brutalist architec...,1472949205,https://live.staticflickr.com/8251/29402571096...,brutalist architecture,flickr
3,36546780916,132394534@N04,radusecatureanu,Brutalist Architecture,beograd belgrade serbia comunism brutalist arc...,1502825597,https://live.staticflickr.com/4339/36546780916...,brutalist architecture,flickr
4,36381698731,73331227@N05,michael.veltman,Brutalist,bruatlist architecture chicago illinois,1502539836,https://live.staticflickr.com/4419/36381698731...,brutalist architecture,flickr


In [23]:
len(df_flickr)

300

In [24]:
df_flickr.to_csv("../data/raw/flickr_metadata_test.csv", index=False)

In [25]:
keywords = [
    "brutalist architecture",
    "concrete building",
    "brutalism facade",
    "modernist concrete building"
]

In [26]:
df_clean = df_flickr.copy()

# 1️⃣ 删除没有 image_url 的
df_clean = df_clean[df_clean["image_url"].notna()]

# 2️⃣ 删除重复（按 URL）
df_clean = df_clean.drop_duplicates(subset=["image_url"])

# 3️⃣ 清理 title（可选）
df_clean["title"] = df_clean["title"].fillna("").str.strip()

# 4️⃣ 清理 tags（统一格式）
df_clean["tags"] = df_clean["tags"].fillna("")

df_clean.head()

,photo_id,owner,owner_name,title,tags,date_upload,image_url,keyword,source
0,43539066400,135414206@N06,Leipzig_trifft_Wien,Brutalist Architecture,brutalist architecture blackandwhite black whi...,1539665544,https://live.staticflickr.com/1922/43539066400...,brutalist architecture,flickr
1,47555029381,34608298@N05,Cat Girl 007,Brutalist Architecture,abstract architecture brutalism brutalist brut...,1554633168,https://live.staticflickr.com/7867/47555029381...,brutalist architecture,flickr
2,29402571096,21224104@N02,Suzanne Neubauer,Brutalist architecture,monochromatic blackandwhite brutalist architec...,1472949205,https://live.staticflickr.com/8251/29402571096...,brutalist architecture,flickr
3,36546780916,132394534@N04,radusecatureanu,Brutalist Architecture,beograd belgrade serbia comunism brutalist arc...,1502825597,https://live.staticflickr.com/4339/36546780916...,brutalist architecture,flickr
4,36381698731,73331227@N05,michael.veltman,Brutalist,bruatlist architecture chicago illinois,1502539836,https://live.staticflickr.com/4419/36381698731...,brutalist architecture,flickr


In [27]:
len(df_clean)

296

In [28]:
df_clean.to_csv("../data/processed/flickr_clean.csv", index=False)

In [29]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [30]:
import os
import requests
from tqdm import tqdm

image_dir = "../data/images"
os.makedirs(image_dir, exist_ok=True)

In [31]:
def download_image(url, folder, idx):
    try:
        r = requests.get(url, timeout=10)
        file_path = os.path.join(folder, f"img_{idx}.jpg")

        with open(file_path, "wb") as f:
            f.write(r.content)

        return file_path
    except:
        return None

In [32]:
downloaded = []

for i, row in tqdm(df_clean.head(100).iterrows(), total=100):
    file_path = download_image(row["image_url"], image_dir, i)

    if file_path:
        downloaded.append(file_path)

100%|██████████| 100/100 [00:51<00:00,  1.94it/s]


In [33]:
len(downloaded)

100

The dataset was cleaned by removing invalid entries and duplicates, ensuring that only valid image URLs and structured metadata were retained for further processing.

In [34]:
import os
from pathlib import Path

image_dir = Path("../data/images")
image_files = sorted([str(p) for p in image_dir.glob("*.jpg")])

print(len(image_files))
print(image_files[:5])

100
['..\\data\\images\\img_0.jpg', '..\\data\\images\\img_1.jpg', '..\\data\\images\\img_10.jpg', '..\\data\\images\\img_11.jpg', '..\\data\\images\\img_12.jpg']


In [35]:
df_images = pd.DataFrame({"local_path": image_files})
df_images.head()

,local_path
0,..\data\images\img_0.jpg
1,..\data\images\img_1.jpg
2,..\data\images\img_10.jpg
3,..\data\images\img_11.jpg
4,..\data\images\img_12.jpg


In [36]:
from PIL import Image

image_info = []

for path in image_files:
    try:
        with Image.open(path) as img:
            image_info.append({
                "local_path": path,
                "width": img.width,
                "height": img.height,
                "mode": img.mode
            })
    except Exception as e:
        image_info.append({
            "local_path": path,
            "width": None,
            "height": None,
            "mode": f"ERROR: {e}"
        })

df_image_info = pd.DataFrame(image_info)
df_image_info.head()

,local_path,width,height,mode
0,..\data\images\img_0.jpg,1023,682,RGB
1,..\data\images\img_1.jpg,1023,683,RGB
2,..\data\images\img_10.jpg,2632,3256,RGB
3,..\data\images\img_11.jpg,1024,734,RGB
4,..\data\images\img_12.jpg,1024,683,RGB


In [37]:
df_image_info["mode"].value_counts()
df_image_info[["width", "height"]].describe()

,width,height
count,100.000000,100.000000
mean,1692.020000,1405.910000
std,1268.228104,1128.325461
min,640.000000,565.000000
25%,1024.000000,683.000000
50%,1024.000000,768.000000
75%,2202.750000,2130.250000
max,5198.000000,5184.000000


In [38]:
import os
from PIL import Image
from pathlib import Path

resize_dir = Path("../data/images_resized")
resize_dir.mkdir(exist_ok=True)

for path in image_files:
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            img.thumbnail((512, 512))
            out_path = resize_dir / Path(path).name
            img.save(out_path, quality=95)
    except:
        pass

The Flickr dataset was cleaned and validated, resulting in 296 usable records. A first batch of 100 images was successfully downloaded for further visual analysis and feature extraction.